# WTMD χN Reweighting Analysis

Compute free energy and probability distribution at different χN values
using histogram reweighting from WTMD simulation data.

**Usage:**
1. Set `chin` (the effective χN of the simulation), `type_pair`, `langevin_step` and `data_dir` below
2. Set the chain model, N and, if the simulation used them, ζN and the smearing. With the grid, the
   box and N̄ saved in the statistics file they give the renormalization factor `z_inf`, computed by
   `scripts/analysis/renormalization.py` (set `polymerfts_dir` to the polymerfts source tree), or set
   `z_inf` to a value of your own
3. Adjust `dchiN` to compute F(Ψ) at different χN values
4. Run all cells

**Reference:**
T. M. Beardsley and M. W. Matsen, J. Chem. Phys. 157, 114902 (2022).

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.interpolate import interp1d
from scipy.signal import find_peaks

In [ ]:
# Parameters - Modify these
chin = 13.15                    # Effective χN = z_inf χ_b N of the simulation (its chi_n input is χ_b N)
type_pair = ("A", "B")          # The two monomer names of that χN
langevin_step = 10_000_000      # Langevin step for statistics file
data_dir = f"lam_{chin:05.2f}/data_simulation"

# χN shift for reweighting (set to 0 for no shift)
dchiN = -0.014

# Renormalization factor z_inf of χ = z_inf χ_b (Matsen et al., J. Chem. Phys. 164, 014905 (2026)).
# It depends on the grid, the box, N̄, the chain model, N, ζN and the smearing: the cell below
# computes it from nx, lx and nbar of the statistics file and these settings of the simulation
# (python scripts/analysis/renormalization.py fields_XXXXXX.mat prints it for a fields file).
polymerfts_dir = "/path/to/polymerfts"  # Source tree with scripts/analysis/renormalization.py
chain_model = "discrete"        # Chain model of the simulation
n_segments = 90                 # N = 1/ds (discrete chains)
zeta_n = None                   # ζN of a compressible model; None: incompressible
smearing = None                 # Smearing input of the simulation, e.g. {"type": "gaussian", "a_int": 0.02}
z_inf = None                    # A value of your own, or None: computed below

print(f"Simulation χN: {chin}")
print(f"Target χN: {chin + dchiN}")

In [ ]:
# Load WTMD statistics
filename = f"{data_dir}/wtmd_statistics_{langevin_step:06d}.mat"
data = loadmat(filename, squeeze_me=True)

psi_range = data['psi_range']
u = data['u']
I0 = data['I0']
dH_psi = data['dH_psi_' + '_'.join(sorted(type_pair))]
dT = data['dT']
dpsi = data['dpsi']

# Renormalization factor on the grid and box of the simulation
if z_inf is None:
    sys.path.insert(0, f"{polymerfts_dir}/scripts/analysis")
    from renormalization import z_inf as renormalization_factor
    z_inf = renormalization_factor(chain_model, np.atleast_1d(data['nx']).tolist(),
                                   np.atleast_1d(data['lx']).tolist(), float(data['nbar']),
                                   n_segments=n_segments, zeta_n=zeta_n, smearing=smearing)
print(f"z_inf = {z_inf:.7f}")

# Compute coefficient and the shift of the bare χ_b N
coeff = 1/dT + 1
bare_dchiN = dchiN / z_inf

# Free energy from bias potential
F = -u * coeff

In [ ]:
# Plot F(Ψ) - Free energy at simulation χN
plt.figure(figsize=(8, 5))
plt.plot(psi_range, F)
plt.xlabel('Ψ')
plt.ylabel('F(Ψ)')
plt.title(f'Free Energy at χN = {chin}')
plt.grid(True)
plt.show()

In [ ]:
# Plot dF/dχN with threshold filtering
plt.figure(figsize=(8, 5))
I0_norm = I0 / np.max(I0)
mask = I0_norm > 1e-1
x = psi_range[mask]
y = dH_psi[mask]

plt.plot(x, y)
plt.xlabel('Ψ')
plt.ylabel(f'∂F/∂(χ[{type_pair[0]}, {type_pair[1]}] N)')
plt.title('Free Energy Derivative (filtered)')
plt.grid(True)
plt.show()

In [ ]:
# Interpolate dH/dχN to full Ψ range
plt.figure(figsize=(8, 5))
interp_func = interp1d(x, y, kind='linear', fill_value='extrapolate')
y_interp = interp_func(psi_range)

plt.plot(psi_range, y_interp)
plt.xlabel('Ψ')
plt.ylabel(f'∂F/∂(χ[{type_pair[0]}, {type_pair[1]}] N) (interpolated)')
plt.title('Interpolated Free Energy Derivative')
plt.grid(True)
plt.show()

In [ ]:
# Compute reweighted free energy at target χN
F_reweighted = F + y_interp * bare_dchiN

plt.figure(figsize=(8, 5))
plt.plot(psi_range, F_reweighted)
plt.xlabel('Ψ')
plt.ylabel('F(Ψ)')
plt.title(f'Reweighted Free Energy at χN = {chin + dchiN:.3f}')
plt.grid(True)
plt.show()

In [ ]:
# Compute probability distribution P(Ψ)
exp_F = np.exp(-F_reweighted)
P_psi = exp_F / np.sum(exp_F) / dpsi

# Find peaks to identify phase coexistence
peaks, _ = find_peaks(-P_psi, prominence=0.01)

if len(peaks) > 0:
    area1 = np.sum(P_psi[:peaks[0]])
    area2 = np.sum(P_psi[peaks[0]:])
    total_area = area1 + area2
    print(f"Phase areas: [{area1:.4f}, {area2:.4f}]")
    print(f"Ordered phase fraction: {area2 / total_area:.4f}")
else:
    print("No significant phase boundary found.")

plt.figure(figsize=(8, 5))
plt.plot(psi_range, P_psi)
plt.xlabel('Ψ')
plt.ylabel('P(Ψ)')
plt.title(f'Probability Distribution at χN = {chin + dchiN:.3f}')
plt.grid(True)
plt.show()